In [1]:
%%capture
%pip install -U accelerate peft trl bitsandbytes transformers
%pip install evaluate sacrebleu rouge-score bert-score

In [1]:
import sys
print(sys.version)

3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


In [5]:
import pkg_resources

# Get version of the libraries used
libraries = ['accelerate', 'peft', 'trl', 'bitsandbytes','pandas', 'numpy','transformers', 'evaluate', 'sacrebleu', 'rouge-score', 'bert-score', 'trl', 'peft', 'datasets']
for lib in libraries:
    try:
        print(f"{lib}: {pkg_resources.get_distribution(lib).version}")
    except pkg_resources.DistributionNotFound:
        print(f"{lib}: Not found")

accelerate: 1.11.0
peft: 0.17.1
trl: 0.24.0
bitsandbytes: 0.48.2
pandas: 2.2.2
numpy: 2.0.2
transformers: 4.57.1
evaluate: 0.4.6
sacrebleu: 2.5.1
rouge-score: 0.1.2
bert-score: 0.3.13
trl: 0.24.0
peft: 0.17.1
datasets: 4.0.0


In [ ]:
from huggingface_hub import login
login(token="")

# Load Model - Configure and Set Up

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import torch

print("Number of GPUs available:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")


In [ ]:
# Import Libraries
# Core
import os
import torch
import pandas as pd
import numpy as np

# Transformers & Tokenization
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback
)
from transformers.trainer_utils import set_seed
from transformers import GenerationConfig
# PEFT
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel
)

# Transformer Reinforcement Learning
from trl import SFTTrainer

# Data
from datasets import Dataset, DatasetDict


set_seed(42) # ensure reproducibility

In [ ]:
 # 4-bit config (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)

model_name = "Qwen/Qwen3-8B"
data_name = model_name.split("/")[-2]

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # safe default for decoder-only

EOS_TOKEN = tokenizer.eos_token  # Must add EOS_TOKEN
tokenizer.padding_side = "left"

# Base model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.config.use_cache = False
model.config.pretraining_tp = 1
model.gradient_checkpointing_enable()

In [ ]:
model.eval()

In [ ]:
# To ensure all adapters saved during checkpointing are used,
# you can explicitly specify the adapter name when loading the fine-tuned model.
# For example, if your adapter is named 'default', you would load it like this:
# fine_model = PeftModel.from_pretrained(model, new_model, adapter_name="default")
# Then you can activate the adapter:
# fine_model.set_adapter("default")

# Evaluate before training

In [ ]:
df = pd.read_json("/content/drive/MyDrive/MIT807/data/silver/question_answers/test.json")
df = df[['instruction', 'question', 'answer', 'context', 'complex_cot', 'options', 'long_answer', 'source', 'category', 'type']]
df.head()

In [ ]:
inference_prompt_style =  """
You are a helpful, respectful, honest, and safe diabetes care assistant.Your answers must be evidence-based, socially unbiased, positive, and emotionally considerate, and must not include harmful, racist, unethical, sexist, toxic, dangerous, or illegal content.
Below is an instruction that describes a task, paired with an input that provides further context.

# Role and Responsibilities:
- Educate medical practitioners, caregivers, and patients using recognized sources.
- Provide decision support (summarize guidelines/risk factors) but never diagnose.
- Map symptoms for referral, explain treatments simply, and recommend South Africa-tailored diet/exercise plans.
- Remind the patient that you are not a substitute for professional care.
- For emergencies, tell the user to seek immediate medical advice from their doctor.
- If a question is unclear or lacks factual coherence, explain why.
- If you don’t know, say so and avoid false information.

# Instruction
{instruction}
{context}
# Question
{question} {options}

# Response {tags}
"""

In [ ]:
def formatting_prompts_func(examples):
    inputs = examples["question"]
    outputs = examples["answer"]
    instructions = examples.get("instruction", [None] * len(inputs)) # Get instructions with default None
    options = examples.get("options", [None] * len(inputs)) # Get options with default None
    contexts = examples.get("context", [None] * len(inputs)) # Get contexts with default None
    cots = examples.get("complex_cot", [None] * len(inputs)) # Get cot with default None
    long_answers = examples.get("long_answer", [None] * len(inputs)) # Get cot with default None

    texts = []
    for question, response, instruction, context, option, cot, long_answer in zip(inputs, outputs, instructions, contexts, options, cots, long_answers):
        if (instruction is None) or (instruction == "") or (instruction == 'nan') or (instruction == "None"):
            instruction = (
                "You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning."
                "Please answer the following medical question."
            )
        if (context == "nan") or (context == "None") or (context is None):
            context = ""
        else:
            context = f"\n# Context:\n{context}\n"

        if (option is None) or (option == "None") or (option == 'nan'):
            option = ""
            tags = "(your response must be detailed, consice, clear and well structured diabetes relevant)"
        else:
            tags = "(your response must always follow this format: [OptionLetter]: [OptionText], then start a new line to provide any additional explanation or other details (if applicable).)"
            option = f"\nOptions: {option}"

        text = inference_prompt_style.format(
            instruction=instruction,
            context=context or "",  # Use empty string if context is None
            question=question,
            tags=tags or "",
            explanation=long_answer,
            options=option, # Use empty string if options is None
        )
        texts.append(text + EOS_TOKEN)

    return {"text": texts}

In [ ]:
from datasets import Dataset

# Convert lists in 'Context' column to strings
df['context'] = df['context'].apply(lambda x: str(x) if isinstance(x, list) else x)

# Convert dictionaries in 'options' column to strings
df['options'] = df['options'].apply(lambda x: str(x) if isinstance(x, dict) else x)

dataset = Dataset.from_pandas(df)
dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
)
print(dataset["text"][1])

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
model.config.use_cache = False

In [ ]:
data_name = "biomistral_qlora"
data_name

In [ ]:
# base model
from tqdm import tqdm
import csv
import torch
import math
import os

BATCH_SIZE = 50#20
CSV_PATH = f"/content/drive/MyDrive/MIT807/data/silver/question_answers/output_new/{data_name}_response_test.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# Resume processing
tokenizer.padding_side = "left"
processed = set()
if os.path.exists(CSV_PATH):
    with open(CSV_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            processed.add(row[0])

dataset_to_process = [x for x in dataset["text"] if x not in processed]

def batch_iterable(iterable, batch_size):
    it = iter(iterable)
    while True:
        batch = []
        try:
            for _ in range(batch_size):
                batch.append(next(it))
        except StopIteration:
            if batch:
                yield batch
            break
        yield batch

with open(CSV_PATH, "a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    total = len(dataset_to_process)
    for batch in tqdm(batch_iterable(dataset_to_process, BATCH_SIZE), desc="Generating (batched)", total=math.ceil(total/BATCH_SIZE)):
        # batch tokenization (adjust to your tokenizer API)
        # Example for HF-style tokenizer + model.generate:
        inputs = tokenizer(batch, return_tensors="pt", padding=True).to(device)

        # faster inference context
        with torch.inference_mode():
            # If your generate_response wraps a custom generation, adapt this:
            outputs = model.generate(
                input_ids=inputs.input_ids,
                max_new_tokens=256,
                attention_mask=inputs.attention_mask,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=True,
                repetition_penalty=1.2,
                do_sample=False
                )
            decoded = tokenizer.batch_decode(outputs, skip_special_tokens=False)
        torch.cuda.empty_cache()
        # stream to disk
        for inp, out in zip(batch, decoded):
            writer.writerow([inp, out])

# PEFT Implementation

In [ ]:
# Prepare for k-bit training (important for QLoRA)
model = prepare_model_for_kbit_training(model,
                                        use_gradient_checkpointing=True,
                                        gradient_checkpointing_kwargs={"use_reentrant": False})

# LoRA config
peft_config = LoraConfig(
    lora_alpha=16,                           # Scaling factor for LoRA
    lora_dropout=0.05,                       # Add slight dropout for regularization
    r=64,                                    # Rank of the LoRA update matrices
    bias="none",                             # No bias reparameterization
    task_type="CAUSAL_LM",                   # Task type: Causal Language Modeling
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],  # Target modules for LoRA
)

model = get_peft_model(model, peft_config)

In [ ]:
output_dir = f"/content/drive/MyDrive/MIT807/{data_name}_qlora/"
os.makedirs(output_dir, exist_ok=True)
# Training Arguments
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=10,   #use grad_accumulation to enlarge effective batch size
    # per_device_eval_batch_size=10,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    max_steps=500,                    # use epochs-based termination; set if you prefer step-based
    learning_rate=1e-5,              # safer starting point; reduce if unstable
    lr_scheduler_type="cosine",      #"linear", #"constant", #recently added
    warmup_ratio=0.05,
    weight_decay=0.001,
    optim="paged_adamw_32bit",        # requires bitsandbytes/paged optimizer in your env
    logging_strategy="steps",
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    bf16=(torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
    fp16=(not torch.cuda.is_bf16_supported() and torch.cuda.is_available()),
    gradient_checkpointing=True,
    report_to=None,
)

# Load Data Training

In [ ]:
train_prompt_style = """
You are a helpful, respectful, honest, and safe diabetes care assistant.Your answers must be evidence-based, socially unbiased, positive, and emotionally considerate, and must not include harmful, racist, unethical, sexist, toxic, dangerous, or illegal content.
Below is an instruction that describes a task, paired with an input that provides further context.

# Role and Responsibilities:
- Educate medical practitioners, caregivers, and patients using recognized sources.
- Provide decision support (summarize guidelines/risk factors) but never diagnose.
- Map symptoms for referral, explain treatments simply, and recommend South Africa-tailored diet/exercise plans.
- Remind the patient that you are not a substitute for professional care.
- For emergencies, tell the user to seek immediate medical advice from their doctor.
- If a question is unclear or lacks factual coherence, explain why.
- If you don’t know, say so and avoid false information.

# Instruction
{instruction}
{context}
# Question
{question} {options}

# Response
{response} {explanation}
"""

In [ ]:
def formatting_prompts_func(examples):
    inputs = examples["question"]
    outputs = examples["answer"]
    instructions = examples.get("instruction", [None] * len(inputs)) # Get instructions with default None
    options = examples.get("options", [None] * len(inputs)) # Get options with default None
    contexts = examples.get("context", [None] * len(inputs)) # Get contexts with default None
    long_answers = examples.get("long_answer", [None] * len(inputs)) # Get cot with default None

    texts = []
    for question, response, instruction, context, option, long_answer in zip(inputs, outputs, instructions, contexts, options, long_answers):
        if (instruction is None) or (instruction == "") or (instruction == 'nan') or (instruction == "None"):
            instruction = (
                "You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning."
                "Please answer the following medical question."
            )

        if (long_answer is None) or (long_answer == "None") or (long_answer == "nan"):
            long_answer = ""
        else:
            long_answer = f"\n{long_answer}"

        if (context == "nan") or (context == "None") or (context is None):
            context = ""
        else:
            context = f"\n# Context:\n{context}\n"

        if (option is None) or (option == "None") or (option == 'nan'):
            option = ""
        else:
            option = f"\nOptions: {option}"

        text = train_prompt_style.format(
            instruction=instruction,
            context=context or "",  # Use empty string if context is None
            question=question,
            explanation=long_answer,
            options=option, # Use empty string if options is None
            response=response,
        )
        texts.append(text + EOS_TOKEN)

    return {"text": texts}

In [ ]:
df = pd.read_json("/content/drive/MyDrive/MIT807/data/silver/question_answers/training.json")
df = df[['instruction', 'question', 'answer', 'context', 'complex_cot', 'category', 'options', 'long_answer']]
df.info()
df.head(1)

In [ ]:
# Convert lists in 'Context' column to strings
df['context'] = df['context'].apply(lambda x: str(x) if isinstance(x, list) else x)

# Convert dictionaries in 'options' column to strings
df['options'] = df['options'].apply(lambda x: str(x) if isinstance(x, dict) else x)

dataset = Dataset.from_pandas(df)
dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
)
print(dataset["text"][1])

In [ ]:
print(dataset["text"][10000])

In [ ]:
#split to test, evaluation and training
dataset = dataset.train_test_split(test_size=0.2, seed=42)
dataset["validation"] = dataset["test"]
del dataset["test"]
#dataset["train"][0] # Removed this line
train_dataset = dataset["train"] # Added this line
eval_dataset = dataset["validation"] # Added this line

In [ ]:
dataset

In [ ]:
train_dataset = dataset["train"]
eval_dataset = dataset["validation"]

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
print(eval_dataset['text'][5])

# Train

In [ ]:
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id

In [ ]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable} / {total} ({100*trainable/total:.4f}%)")

In [ ]:
from transformers import TrainerCallback
class SaveAtEpochEndCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        # this flag tells Trainer to save at the end of each epoch
        control.should_save = True
        return control

In [ ]:
# Initialize the Trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset, # Corrected line
    data_collator=data_collator,
    eval_dataset=eval_dataset, # Added this line
    callbacks=[
        SaveAtEpochEndCallback
    ]
)

In [ ]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable} / {total} ({100*trainable/total:.4f}%)")

In [ ]:
# train
import gc, torch
gc.collect()
torch.cuda.empty_cache()
model.config.use_cache = False
trainer.train(resume_from_checkpoint=True)

TrainOutput(global_step=500, training_loss=1.0312340850830077, metrics={'train_runtime': 6694.5925, 'train_samples_per_second': 1.494, 'train_steps_per_second': 0.075, 'total_flos': 3.832637717282611e+17, 'train_loss': 1.0312340850830077, 'epoch': 0.7757951900698216})

In [ ]:
# Save new model
new_model = "/content/drive/MyDrive/MIT807/qwen_qlora_fine_tuned/"
trainer.model.save_pretrained(new_model)
data_name = new_model.split("/")[-2]
data_name

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
model.config.use_cache = False

# Evaluate performance

In [ ]:
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel
)

In [ ]:
fine_model = PeftModel.from_pretrained(model, new_model)
# Move the model to bfloat16 after loading
fine_model.to(torch.bfloat16)
# Reload tokenizer to save it
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
print(dataset["text"][2])

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
model.config.use_cache = False

In [ ]:
from tqdm import tqdm
import csv
import torch
import math
import os

BATCH_SIZE = 50#20
CSV_PATH = f"/content/drive/MyDrive/MIT807/data/silver/question_answers/output_new/{data_name}_response_test_v2.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer.padding_side = "left"
fine_model.to(device) # Ensure fine_model is on the correct device
fine_model.eval()

# Resume processing
processed = set()
if os.path.exists(CSV_PATH):
    with open(CSV_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        # Check if the file is not empty before trying to read the header
        if os.path.getsize(CSV_PATH) > 0:
            try:
                next(reader) # Skip header row if exists
            except StopIteration:
                pass  # Handle empty file or file with only header


            for row in reader:
                # Ensure row has enough elements before accessing row[0]
                if row and len(row) > 0:
                    processed.add(row[0])


dataset_to_process = [x for x in dataset["text"] if x not in processed]

def batch_iterable(iterable, batch_size):
    it = iter(iterable)
    while True:
        batch = []
        try:
            for _ in range(batch_size):
                batch.append(next(it))
        except StopIteration:
            if batch:
                yield batch
            break
        yield batch

with open(CSV_PATH, "a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    total = len(dataset_to_process)
    for batch in tqdm(batch_iterable(dataset_to_process, BATCH_SIZE), desc="Generating (batched)", total=math.ceil(total/BATCH_SIZE)):
        # batch tokenization (adjust to your tokenizer API)
        inputs = tokenizer(batch, return_tensors="pt", padding=True).to(device)

        # No explicit casting needed here if fine_model is already in bfloat16

        # faster inference context
        with torch.inference_mode():
            # If your generate_response wraps a custom generation, adapt this:
            outputs = fine_model.generate(
                input_ids=inputs["input_ids"],
                max_new_tokens=256,
                attention_mask=inputs["attention_mask"],
                eos_token_id=tokenizer.eos_token_id,
                use_cache=True,
                repetition_penalty=1.2,
                do_sample=False
                )
            decoded = tokenizer.batch_decode(outputs, skip_special_tokens=False)
        torch.cuda.empty_cache()
        # stream to disk
        for inp, out in zip(batch, decoded):
            writer.writerow([inp, out])

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
model.config.use_cache = False

In [ ]:
# idle for 5 second
import time
time.sleep(5)